In [1]:
!pip install -q librosa tqdm pandas

import os
import re
import gc
import numpy as np
import librosa
import warnings
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import pandas as pd
import psutil

warnings.filterwarnings('ignore')

BASE_PATH  = "/kaggle/input/datasets/alieldinalaa/nn-cmp27-dataset"
INPUT_DIR  = BASE_PATH
OUTPUT_DIR = "/kaggle/working/Feature-Extraction-Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input  : {INPUT_DIR}")
print(f"Output : {OUTPUT_DIR}")
print("Setup done ✅")

Input  : /kaggle/input/datasets/alieldinalaa/nn-cmp27-dataset
Output : /kaggle/working/Feature-Extraction-Output
Setup done ✅


In [2]:
# ══════════════════════════════════════════════════════════════════════
# PREPROCESSING  — exact copy from 01_preprocessing/preprocessin.kaggle.py
# ══════════════════════════════════════════════════════════════════════

def preprocess_audio(file_path, target_sr=16000, chunk_duration=3.0, step_duration=1.5):
    y, sr = librosa.load(file_path, sr=target_sr, res_type='soxr_hq')

    y_trimmed, _ = librosa.effects.trim(y, top_db=20)

    max_amp = np.max(np.abs(y_trimmed))
    if max_amp > 0:
        y = y_trimmed / max_amp
    else:
        y = y_trimmed

    chunk_length = int(target_sr * chunk_duration)
    step_length  = int(target_sr * step_duration)

    chunks = []

    if len(y) < chunk_length:
        y = np.pad(y, (0, chunk_length - len(y)))
        chunks.append(y)
    else:
        start = 0
        while start + chunk_length <= len(y):
            chunks.append(y[start:start + chunk_length])
            start += step_length

        if start < len(y):
            chunks.append(y[-chunk_length:])

    return chunks

In [3]:
# ══════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION  — Log-Mel Spectrogram
# ══════════════════════════════════════════════════════════════════════

def extract_mel_spectrograms(chunks, sr=16000, n_fft=1024, hop_length=512, n_mels=128):
    features = []
    for chunk in chunks:
        mel_spec = librosa.feature.melspectrogram(
            y=np.asarray(chunk, dtype=np.float32),
            sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
        )
        log_mel = librosa.power_to_db(mel_spec, ref=1.0)
        features.append(log_mel)
    return np.array(features, dtype=np.float32)   # (num_chunks, 128, 94)

In [4]:
# ══════════════════════════════════════════════════════════════════════
# LABEL MAPPING  (6-class as per project document)
# ══════════════════════════════════════════════════════════════════════

LABEL_MAP = {
    (1, "Normal"):   0,  (1, "Abnormal"): 1,
    (2, "Normal"):   2,  (2, "Abnormal"): 3,
    (3, "Normal"):   4,  (3, "Abnormal"): 5,
}
LABEL_NAMES = {
    0: "Machine 1 – Normal",   1: "Machine 1 – Abnormal",
    2: "Machine 2 – Normal",   3: "Machine 2 – Abnormal",
    4: "Machine 3 – Normal",   5: "Machine 3 – Abnormal",
}

def get_label(machine_folder, status):
    match = re.search(r'\d+', machine_folder)
    if not match:
        raise ValueError(f"Cannot parse machine number from: {machine_folder}")
    key = (int(match.group()), status)
    if key not in LABEL_MAP:
        raise ValueError(f"Unknown combination: {key}")
    return LABEL_MAP[key]

In [5]:
# ══════════════════════════════════════════════════════════════════════
# TASK GATHERING
# ══════════════════════════════════════════════════════════════════════

tasks = []

for machine in sorted(os.listdir(INPUT_DIR)):
    machine_path = os.path.join(INPUT_DIR, machine)
    if not os.path.isdir(machine_path):
        continue

    machine_out_dir = os.path.join(OUTPUT_DIR, machine)
    os.makedirs(machine_out_dir, exist_ok=True)

    data_path = os.path.join(machine_path, "machine_data")
    if not os.path.exists(data_path):
        continue

    for label_name in ["Normal", "Abnormal"]:
        class_path = os.path.join(data_path, label_name)
        if not os.path.exists(class_path):
            continue

        label = get_label(machine, label_name)

        for file in sorted(os.listdir(class_path)):
            if file.endswith(".wav"):
                tasks.append({
                    "file_path":       os.path.join(class_path, file),
                    "label":           label,
                    "machine":         machine,
                    "label_name":      label_name,
                    "base_name":       os.path.splitext(file)[0],
                    "machine_out_dir": machine_out_dir,
                })

print(f"Total audio files: {len(tasks)}")

Total audio files: 56236


In [6]:
# ══════════════════════════════════════════════════════════════════════
# WORKER — preprocess → extract → save
# ══════════════════════════════════════════════════════════════════════

def process_file(task):
    file_path       = task["file_path"]
    label           = task["label"]
    machine         = task["machine"]
    label_name      = task["label_name"]
    base_name       = task["base_name"]
    machine_out_dir = task["machine_out_dir"]

    save_name = f"{machine}_{label_name}_{base_name}.npz"
    save_path = os.path.join(machine_out_dir, save_name)

    # Resume: skip already-extracted files
    if os.path.exists(save_path):
        try:
            with np.load(save_path) as d:
                return (save_name, label, machine, d["features"].shape[0], save_path)
        except Exception:
            os.remove(save_path)

    try:
        # Step 1 — Preprocessing
        chunks = preprocess_audio(file_path)

        # Step 2 — Feature extraction
        features = extract_mel_spectrograms(chunks)  # (N, 128, 94)

        # Step 3 — Save (uncompressed — disk is fine, skip slow compression)
        np.savez(save_path, features=features)

        n = features.shape[0]
        del chunks, features

        return (save_name, label, machine, n, save_path)

    except Exception as e:
        print(f"❌ {file_path}: {e}")
        return None

In [7]:
# ══════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════

all_results = []

with Pool(cpu_count()) as pool:
    for i, res in enumerate(tqdm(pool.imap_unordered(process_file, tasks), total=len(tasks))):
        if res is not None:
            all_results.append(res)
            
        # Log RAM and Disk usage every 100 files
        if (i + 1) % 100 == 0:
            ram = psutil.virtual_memory()
            disk = psutil.disk_usage('/kaggle/working')
            tqdm.write(f"  [File {i+1}/{len(tasks)}] "
                       f"RAM: {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f} GB | "
                       f"Disk: {disk.used/1024**3:.1f}/{disk.total/1024**3:.1f} GB")

print(f"✅ Done — {len(all_results)} / {len(tasks)} files processed")

  0%|          | 100/56236 [00:41<1:32:48, 10.08it/s]

  [File 100/56236] RAM: 1.4/31.4 GB | Disk: 0.0/19.5 GB


  0%|          | 200/56236 [00:52<1:41:22,  9.21it/s]

  [File 200/56236] RAM: 1.4/31.4 GB | Disk: 0.1/19.5 GB


  1%|          | 301/56236 [01:04<1:37:05,  9.60it/s]

...

  [File 56100/56236] RAM: 1.4/31.4 GB | Disk: 10.9/19.5 GB


100%|█████████▉| 56202/56236 [1:26:18<00:03, 11.08it/s]

  [File 56200/56236] RAM: 1.4/31.4 GB | Disk: 10.9/19.5 GB


100%|██████████| 56236/56236 [1:26:21<00:00, 10.85it/s]

✅ Done — 56236 / 56236 files processed


In [8]:
# ══════════════════════════════════════════════════════════════════════
# METADATA + SUMMARY
# ══════════════════════════════════════════════════════════════════════

df = pd.DataFrame(all_results, columns=["file", "label", "machine", "num_chunks", "file_path"])
df.to_csv(os.path.join(OUTPUT_DIR, "metadata.csv"), index=False)

print(f"Total files   : {len(df)}")
print(f"Total chunks  : {int(df['num_chunks'].sum())}")
print("\nLabel distribution:")
for lbl, count in df["label"].value_counts().sort_index().items():
    print(f"  {lbl}  {LABEL_NAMES[lbl]:28s} → {count} files")

Total files   : 56236
Total chunks  : 241549

Label distribution:
  0  Machine 1 – Normal           → 16200 files
  1  Machine 1 – Abnormal         → 3176 files
  2  Machine 2 – Normal           → 16200 files
  3  Machine 2 – Abnormal         → 3240 files
  4  Machine 3 – Normal           → 14400 files
  5  Machine 3 – Abnormal         → 3020 files


In [9]:
# ══════════════════════════════════════════════════════════════════════
# OUTPUT CHECK — verify every saved .npz is readable with no errors
# ══════════════════════════════════════════════════════════════════════

print("Running output integrity check …\n")

errors   = []
ok_count = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking"):
    fpath = row["file_path"]
    try:
        with np.load(fpath) as data:
            feats = data["features"]
            assert feats.ndim == 3,                  "Expected 3D array"
            assert feats.shape[1] == 128,            "Expected 128 mel bins"
            assert not np.isnan(feats).any(),        "Contains NaN"
            assert not np.isinf(feats).any(),        "Contains Inf"
        ok_count += 1
    except Exception as e:
        errors.append((row["file"], str(e)))

print(f"\n  ✅ OK     : {ok_count}")
print(f"  ❌ Errors : {len(errors)}")

if errors:
    print("\n  Failed files:")
    for fname, err in errors:
        print(f"    {fname}: {err}")
else:
    print("\n✅ All outputs verified — no errors found!")

Running output integrity check …



Checking: 100%|██████████| 56236/56236 [02:26<00:00, 383.94it/s] 


  ✅ OK     : 56236
  ❌ Errors : 0

✅ All outputs verified — no errors found!
